# Week 3 Day 5 challenge: a discount agent on AgentCore Runtime

Northwind Software's discount agent approves discounts up to 15% on its own; above that a sales director signs off.
The write-up is in `README.md` beside this notebook. This notebook runs the tests, deploys the agent, sends it five
requests on Runtime, shows each rep's notes in AgentCore Memory, and tears it all down.

You need `OPENROUTER_API_KEY` and `AWS_DATABASE_URL` in `.env`, and to be signed in to AWS. Runs at most 5 requests.

In [ ]:
import truststore

truststore.inject_into_ssl()

import json
import subprocess
import sys
import time
from pathlib import Path

from dotenv import dotenv_values

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
env = dotenv_values(HERE.parents[2] / ".env")

import deploy

## The tests

A scripted model, checkpoints in memory: no model calls, no AWS.

In [ ]:
print(subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--no-cov", "-p", "no:cacheprovider"],
                     capture_output=True, text=True).stdout[-300:])

## Deploy

Packages the code for Linux on ARM64, then makes the bucket, the two credentials, the Memory, the role and the runtime. About 3 minutes.

In [ ]:
arn, memory_id = deploy.deploy(env["OPENROUTER_API_KEY"], env["AWS_DATABASE_URL"])
print(arn)
print(memory_id)

## Five requests

Each call is a new Runtime session. The sign-offs resume the paused request from its checkpoint in Postgres.

In [ ]:
def show(r):
    print(json.dumps({k: r.get(k) for k in ("status", "decision", "signed_by", "refused", "model_calls") if r.get(k) is not None}, indent=1))
    return r

base = {"operation": "request", "seats": 150}
show(deploy.invoke(arn, base | {"rep_id": "REP-01", "customer_id": "BLUEFIN", "sku": "ANALYTICS", "percent": 10,
    "justification": "Expanding to three regions, 150 seats.",
    "note": "I like the percent first, then a one-line reason. No more than one sentence."}))

In [ ]:
held = show(deploy.invoke(arn, base | {"rep_id": "REP-01", "customer_id": "ACME", "sku": "SECURITY", "percent": 22,
    "justification": "Competitive replacement: Contoso quoted 20% under list on a 3-year deal."}))

In [ ]:
show(deploy.invoke(arn, {"operation": "sign_off", "thread_id": held["thread_id"], "person_id": "REP-02", "answer": "approve"}))
show(deploy.invoke(arn, {"operation": "sign_off", "thread_id": held["thread_id"], "person_id": "DIR-01", "answer": "approve"}))

## Memory, a namespace per rep

AgentCore Memory turns REP-01's requests into notes in a minute or two. REP-02's namespace stays empty.

In [ ]:
for _ in range(18):
    if deploy.notes(memory_id, "REP-01"):
        break
    time.sleep(20)
print("REP-01:", deploy.notes(memory_id, "REP-01"))
print("REP-02:", deploy.notes(memory_id, "REP-02"))

In [ ]:
show(deploy.invoke(arn, base | {"rep_id": "REP-01", "customer_id": "CEDAR", "sku": "SUPPORT", "percent": 6,
    "justification": "Two-year renewal."}))

## Tear down

Deletes the runtime, the Memory, the credentials, the role and the bucket. The checkpoints stay in the `discounts` schema on RDS until the RDS teardown.

In [ ]:
deploy.tear_down()